<div style="border-left:6px solid #00C98B;padding:10px 18px;margin-bottom:18px">
<h1 style="margin:0">Solving a QUBO with QoolQit &mdash; Hands-on</h1>
<p style="margin:6px 0 0;color:#52666d"><strong>Guided exercise</strong> &middot; From an optimization model to sampled candidate solutions</p>
</div>

In this notebook you will take a small QUBO all the way to sampled candidate solutions:

- the QUBO and its exact classical solution are **given** (section 1 and the analysis helpers of
  section 6) &mdash; run those cells as they are;
- **you** embed the pairwise coefficients into a neutral-atom register, build a dimensionless
  adiabatic drive, compile it to a device and run it on a local emulator, then measure how often
  the optimal bitstrings are sampled.

### How to work through this notebook

The hands-on cells are partially written. Wherever you see `...` you have to fill something in,
and the comment right above tells you what. The tasks are labelled:

| Label | Meaning |
|---|---|
| **Task** | fill in the `...` to make the cell run |
| **Question** | no code needed &mdash; write your answer in the markdown cell below |
| **Bonus** | optional, do it if you have time left |

Cells marked *given* are complete: they set up the classical side of the problem (the cost
function, the exhaustive reference solution, the metrics) so that the exercise stays focused on
the quantum part.

<b>Hints</b> are hidden in collapsible blocks.


----

## 1. Define the QUBO

We minimize a quadratic cost over binary variables $x_i\in\{0,1\}$:

$$
C(x)=\sum_i Q_{ii}x_i+\sum_{i<j}Q_{ij}x_ix_j
$$

This is a **vehicle routing** instance: four vans leave a depot and each picks one of three
delivery plans (north, centre, south). The variable $x_{vp}=1$ means van $v$ takes plan $p$, and the
twelve variables are ordered `1N, 1C, 1S, 2N, 2C, 2S, 3N, 3C, 3S, 4N, 4C, 4S`, so the coefficient
matrix reads

$$
Q=\begin{pmatrix}
-35 & 100 & 90  & 20  & 10  & 0   & 20  & 10  & 0   & 20  & 10  & 0   \\
0   & -35 & 90  & 10  & 20  & 0   & 10  & 20  & 0   & 10  & 20  & 0   \\
0   & 0   & -35 & 0   & 0   & 20  & 0   & 0   & 20  & 0   & 0   & 20  \\
0   & 0   & 0   & -35 & 100 & 90  & 20  & 10  & 0   & 20  & 10  & 0   \\
0   & 0   & 0   & 0   & -35 & 90  & 10  & 20  & 0   & 10  & 20  & 0   \\
0   & 0   & 0   & 0   & 0   & -35 & 0   & 0   & 20  & 0   & 0   & 20  \\
0   & 0   & 0   & 0   & 0   & 0   & -35 & 100 & 90  & 20  & 10  & 0   \\
0   & 0   & 0   & 0   & 0   & 0   & 0   & -35 & 90  & 10  & 20  & 0   \\
0   & 0   & 0   & 0   & 0   & 0   & 0   & 0   & -35 & 0   & 0   & 20  \\
0   & 0   & 0   & 0   & 0   & 0   & 0   & 0   & 0   & -35 & 100 & 90  \\
0   & 0   & 0   & 0   & 0   & 0   & 0   & 0   & 0   & 0   & -35 & 90  \\
0   & 0   & 0   & 0   & 0   & 0   & 0   & 0   & 0   & 0   & 0   & -35
\end{pmatrix}
$$

Read the matrix as three kinds of terms:

- the **diagonal** $-35$ is the reward for assigning a van to *some* plan;
- the **large in-block terms** ($100$, $90$) penalise a van taking two plans at once;
- the **small cross-block terms** ($20$, $10$, $0$) are the congestion cost of two different vans
  choosing overlapping plans.

The cell below types this matrix in as `Q_exp` and symmetrizes it into `Q`, the form QoolQit
expects. This section is *given* &mdash; just run it.


In [ ]:
from collections import Counter
from itertools import product

import matplotlib.pyplot as plt
import numpy as np

# Upper-triangular coefficient matrix of the instance described above.
Q_exp = np.array(
    [
        [-35, 100, 90, 20, 10, 0, 20, 10, 0, 20, 10, 0],
        [0, -35, 90, 10, 20, 0, 10, 20, 0, 10, 20, 0],
        [0, 0, -35, 0, 0, 20, 0, 0, 20, 0, 0, 20],
        [0, 0, 0, -35, 100, 90, 20, 10, 0, 20, 10, 0],
        [0, 0, 0, 0, -35, 90, 10, 20, 0, 10, 20, 0],
        [0, 0, 0, 0, 0, -35, 0, 0, 20, 0, 0, 20],
        [0, 0, 0, 0, 0, 0, -35, 100, 90, 20, 10, 0],
        [0, 0, 0, 0, 0, 0, 0, -35, 90, 10, 20, 0],
        [0, 0, 0, 0, 0, 0, 0, 0, -35, 0, 0, 20],
        [0, 0, 0, 0, 0, 0, 0, 0, 0, -35, 100, 90],
        [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, -35, 90],
        [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, -35],
    ],
    dtype=float,
)

# QoolQit expects a *symmetric* coefficient matrix, while we typed only the upper
# triangle. Symmetrizing leaves the cost function unchanged: x @ Q @ x == C(x).
Q = (Q_exp + Q_exp.T) / 2

N = len(Q)
print(f"Problem size: {N} binary variables ({2**N} candidate bitstrings)")


### Classical reference solution &mdash;

Before touching any quantum hardware we solve the instance *classically*, by brute force. With
$N=12$ there are only $4096$ bitstrings, so an exhaustive enumeration is instant and gives the
exact reference against which we will later score the quantum samples:

- `best_cost`: the minimum value of $C(x)$;
- `optimal_solutions`: **all** bitstrings that reach it.

The cell below enumerates every bitstring, evaluates $C(x)=x^\top Q x$ and keeps the minimizers.
Just run it and move on to section 2, where the hands-on work starts.


In [ ]:
def qubo_cost(bitstring, q):
    """Cost C(x) = x^T q x, with q the symmetrized coefficient matrix."""
    x = np.fromiter((int(b) for b in bitstring), dtype=int)
    return float(x @ q @ x)


# Exhaustive enumeration of every candidate bitstring.
all_costs = {"".join(bits): qubo_cost("".join(bits), Q) for bits in product("01", repeat=N)}

best_cost = min(all_costs.values())
optimal_solutions = np.array(
    sorted(bits for bits, cost in all_costs.items() if cost == best_cost)
)

print(f"best_cost = {best_cost}")
print(f"number of optimal solutions = {len(optimal_solutions)}")
print(optimal_solutions)


The scan returns `best_cost = -110.0`, reached by **12** bitstrings. All of them assign exactly
one plan per van, and all of them are permutations of the plan multiset
$\{\text{N},\text{C},\text{S},\text{S}\}$: with four vans and three plans one plan has to be
served twice, and the cheapest choice is to double up on the south plan. Since the vans are
interchangeable, every relabelling of the four vans gives another optimum, hence
$4!/2!=12$ degenerate ground states.

Keep this in mind for section 6: a good quantum sample is one that lands on *any* of these 12
bitstrings.


## 2. Embed the QUBO into an atom register

A Rydberg register cannot realise arbitrary coefficients: the interaction between two atoms is
fixed by their distance ($\propto 1/r^6$) and is always positive. The **embedder** searches for
atom positions whose interaction matrix is as close as possible to the target off-diagonal
coefficients.

Two things matter here:

1. the coefficients must be **rescaled to dimensionless units** before embedding &mdash; QoolQit works
   in dimensionless time/energy and the compilation step later maps them onto the device;
2. only the **off-diagonal** part is embedded; the diagonal becomes the detuning in section 3.

**Task 2.1** &mdash; normalise `Q` by its largest coefficient and embed it with `Blade`.

<details><summary>Hint</summary>

`interaction_scale = np.max(Q)`, then `Q_norm = Q / interaction_scale`.
`Blade(BladeConfig())` gives the default embedder; `embedder.embed(Q_norm)` returns a graph
that you can `.draw()`.
</details>


In [ ]:
from qoolqit.embedding import Blade, BladeConfig

# Task 2.1a: rescale the coefficients so the largest one becomes 1.
interaction_scale = ...
Q_norm = ...

# Task 2.1b: instantiate the embedder with its default configuration and embed Q_norm.
embedder = ...
embedded_graph = ...

embedded_graph.draw()


**Task 2.2** &mdash; the embedding is approximate, so quantify the error. For every embedded pair
$(i,j)$ compare the realised interaction with the target `Q_norm[i, j]` and report the mean
absolute error.


In [ ]:
# Summarize embedding quality
embedding_rows = []
for (i, j), embedded_value in embedded_graph.interactions().items():
    target_value = ...                       # the coefficient we asked for
    error = ...                              # absolute deviation
    embedding_rows.append((i, j, target_value, embedded_value, error))

# Sort by the magnitude of the *target* coupling: the large ones are the ones we care about.
embedding_rows.sort(key=lambda row: abs(row[2]), reverse=True)
errors = np.array([row[4] for row in embedding_rows])

print(f"Mean absolute embedding error:    {...:.4f}")
print("\nCoupling errors:")
for i, j, target, embedded, error in embedding_rows[::4]:   # print every 4th row
    print(
        f"  ({i}, {j})  target={target:>7.4f}  "
        f"embedded={embedded:>7.4f}  |error|={error:.4f}"
    )


**Question 2.3** &mdash; look at the pairs whose *target* coupling is exactly `0`. Are they embedded
faithfully? Why can a fully connected Rydberg system never realise an exact zero, and what does
the embedder do instead?

> Exact zero interactions cannot generally be realized in a fully connected Rydberg system; the
> embedder makes them small by placing the corresponding atoms farther apart.

**Task 2.4** &mdash; turn the embedded graph into a QoolQit `Register` and draw it.


In [ ]:
from qoolqit import Register

# Task: build the register from the embedded graph (there is a dedicated constructor).
register = ...
register.draw()


## 3. Writing the adiabatic drive

The drive is the annealing schedule. We start in the ground state of the initial Hamiltonian
(all atoms in $|g\rangle$, strongly negative detuning) and slowly rotate towards the final
Hamiltonian whose ground state encodes the QUBO optimum.

Two waveforms:

- the **amplitude** $\Omega(t)$ opens and closes smoothly: it must vanish at $t=0$ and $t=T$ so
  that the initial and final Hamiltonians are diagonal in the computational basis;
- the **detuning** $\delta(t)$ ramps from negative to positive, sweeping the system across the
  transition. Its final value must reproduce the QUBO diagonal &mdash; note the **minus sign**: the
  Rydberg Hamiltonian has $-\delta\, n_i$, so a *reward* $Q_{ii}<0$ becomes a *positive* detuning.

**Task 3.1** &mdash; choose the scales and build the drive:

- `omega_max`: the mean of the strictly positive off-diagonal couplings of `Q_norm`
  (an energy scale comparable to the interactions we must compete with);
- `delta_final`: the largest of `-diag(Q_norm)`;
- `delta_initial = -delta_final`, so the sweep is symmetric;
- `T = 50.0` dimensionless time units;
- the amplitude as an `InterpolatedWaveform` through `[0, omega_max, omega_max, 0]` at relative
  times `[0, 0.25, 0.75, 1]` (a trapezoid), the detuning as a `RampWaveform`.

<details><summary>Hint</summary>

`np.triu_indices(N, k=1)` picks the strict upper triangle; then keep only entries `> 0`.
</details>


In [ ]:
from qoolqit import Drive
from qoolqit.waveforms import InterpolatedWaveform, RampWaveform

# Task 3.1a: collect the strictly positive off-diagonal couplings and average them.
positive_couplings = Q_norm[...]
positive_couplings = positive_couplings[...]
omega_max = float(...)

# Task 3.1b: the final detuning has to reproduce the QUBO diagonal (mind the sign!).
delta_targets = ...
delta_final = float(...)
delta_initial = ...

# Total (dimensionless) duration of the sweep.
T = 50.0

# Task 3.1c: trapezoidal amplitude, 0 -> omega_max -> omega_max -> 0.
amplitude = InterpolatedWaveform(
    T,
    values=...,
    times=...,
)

# Task 3.1d: linear detuning ramp from delta_initial to delta_final.
detuning = RampWaveform(...)

drive = Drive(amplitude=amplitude, detuning=detuning)
drive.draw()

print(f"omega_max     = {omega_max:.4f}")
print(f"delta_initial = {delta_initial:.4f}")
print(f"delta_final   = {delta_final:.4f}")
print(f"duration T    = {T:.1f}")


**Question 3.2** &mdash; why does the amplitude have to start *and* end at zero? What would you
expect to see in the sampled bitstrings if it ended at a finite value?


## 4. Compile

So far everything was dimensionless. Compilation maps the abstract program onto a concrete
device: it rescales time and energy to physical units (ns, rad/µs) and rescales the register to
µm, checking every hardware constraint on the way.

The `profile` decides *how* the free scaling factor is fixed. `"max_energy"` pushes the
interaction energy as high as the device allows, which keeps the sweep short.

**Task 4.1** &mdash; build the `QuantumProgram`, compile it to `AnalogDevice()` with the
`"max_energy"` profile, and draw the resulting sequence.


In [ ]:
from qoolqit import AnalogDevice, QuantumProgram

# Task: pair the register with the drive, then compile onto the analog device.
program = QuantumProgram(register=..., drive=...)
program.compile_to(device=..., profile=...)
program.compiled_sequence.draw()


**Question 4.2** &mdash; compare the drawn sequence with `drive.draw()` from section 3. Which axes
changed and which shapes stayed the same? Read off the physical duration of the pulse.


## 5. Execute

**Task 5.1** &mdash; run the compiled program on the local emulator with `10000` shots, collect the
final bitstrings into a `Counter`, and print the five most frequent ones.

Do any of them appear in your `optimal_solutions` list?


In [ ]:
from qoolqit.execution import LocalEmulator

# Task: instantiate the emulator, run the program and count the measured bitstrings.
emulator = ...
job = ...
results = job.results()
counts_single = Counter(...)          

print("Five most frequently sampled bitstrings:")
for bits, count in counts_single.most_common(5):
    flag = "  <-- optimal!" if bits in set(optimal_solutions) else ""
    print(f"  {bits}: {count}{flag}")


**Task 5.2** &mdash; a single run tells us little. The compilation profile leaves one knob free:
`device_max_duration_ratio`, i.e. what fraction of the device's maximum pulse duration we use.
Short sweeps are less adiabatic, long sweeps suffer more from noise and decay.

Loop over the ratios below, recompile the *same* register and drive for each one, run 1000 shots
locally, and store one `Counter` per ratio in the list `counts`.

<details><summary>Hint</summary>

Rebuild `QuantumProgram(register=register, drive=drive)` inside the loop &mdash; a program that has
already been compiled should not be recompiled with different settings.
`program.compile_to(device=AnalogDevice(), profile="max_energy", device_max_duration_ratio=ratio)`
</details>


In [ ]:
ratios = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0]

counts = []
for ratio in ratios:
    # Task 5.2a: a fresh program for every ratio ...
    program = ...
    # Task 5.2b: ... compiled with this duration ratio ...
    program.compile_to(device=..., profile=..., device_max_duration_ratio=...)

    # Task 5.2c: ... and run with 1000 shots on the local emulator.
    local_emulator = LocalEmulator(num_shots=...)
    job_local = ...

    results = job_local.results()
    counts.append(Counter(results.final_bitstrings))

print(f"collected {len(counts)} count distributions")


<details><summary><b>Bonus 5.3</b> &mdash; run the same sweep on the cloud (needs credentials)</summary>

If you have a Pasqal Cloud account, export `CLOUD_USERNAME`, `CLOUD_PASSWORD` and
`CLOUD_PROJECT` in your environment and fill in the cell below. Cloud jobs are queued, so
submit them now and come back to section 7 to collect the results.

The cell also writes the `job_id` / `batch_id` of every submission to `my_cloud_jobs.json`.
Keep that file: it is the only way to find your jobs again in section 7, and you will need it
if the kernel restarts (or if you close the notebook) before the queue gets to them.

</details>


In [ ]:
# Bonus: submit the same sweep to the cloud. Uncomment and complete if you have credentials.
#
# import json
# import os
# from pathlib import Path
#
# from pasqal_cloud import PasqalCloudConnection
# from qoolqit.execution import RemoteEmulator, get_batch_id
#
# connection = PasqalCloudConnection(
#     username=os.getenv("CLOUD_USERNAME"),
#     password=os.getenv("CLOUD_PASSWORD"),
#     project_id=os.getenv("CLOUD_PROJECT"),
# )
#
# JOBS_CACHE_PATH = Path("my_cloud_jobs.json")
#
# cloud_jobs = []
# cloud_job_records = []
# for ratio in ratios:
#     program = ...
#     program.compile_to(device=..., profile=..., device_max_duration_ratio=ratio)
#     remote_emulator = RemoteEmulator(num_shots=1000, connection=connection)
#     job_cloud = remote_emulator.run(program)
#     cloud_jobs.append(job_cloud)
#
#     # Cloud jobs are queued, so the results are not ready yet. Record the identifiers of
#     # every submitted job -- section 7 needs them to fetch the results later, and without
#     # them a kernel restart loses the jobs for good.
#     cloud_job_records.append(
#         {
#             "ratio": ratio,
#             "job_id": job_cloud.job_id(),
#             "batch_id": get_batch_id(job_cloud),
#         }
#     )
#
# # Write the identifiers to disk *after* the loop, so section 7 works in a fresh kernel too.
# with JOBS_CACHE_PATH.open("w") as f:
#     json.dump(cloud_job_records, f, indent=2)
#
# print(f"saved {len(cloud_job_records)} job identifiers to {JOBS_CACHE_PATH}")
# for record in cloud_job_records:
#     print(f"  ratio={record['ratio']:.1f}  job={record['job_id']}  batch={record['batch_id']}")


## 6. Analyse the sampled solutions

Raw counts are hard to read. We score each distribution with four numbers:

| Metric | Meaning | Good value |
|---|---|---|
| `avg_gap_cost` | shot-averaged distance of $C(x)$ from the optimum | low |
| `optimality_gap` | how far the *best* sampled string is from the optimum | 0 |
| `feasible_fraction` | fraction of shots respecting the one-plan-per-van constraint | high |
| `success_probability` | fraction of shots that *are* an optimal solution | high |

These are purely classical post-processing, so the next two cells are *given*: run them, then
use them in Task 6.4. Note `is_feasible_vrp`: the bitstring splits into blocks of
`group_size = 3` (one per van) and a shot is feasible when every block contains exactly one `1`.


In [ ]:
def success_probability(counter, optimal):
    """Fraction of shots that landed on one of the known optimal bitstrings."""
    counter = Counter(counter)
    total = sum(counter.values())
    return sum(counter.get(bits, 0) for bits in optimal) / total


def is_feasible_vrp(bitstring: str, group_size: int = 3) -> bool:
    """A shot is feasible if every van (block of `group_size` bits) picks exactly one plan
    (i.e. no van is left idle and no van is assigned to more than one path)."""
    return all(
        bitstring[i : i + group_size].count("1") == 1
        for i in range(0, len(bitstring), group_size)
    )


# Quick self-check.
assert is_feasible_vrp("001001010100")          # one plan per van
assert not is_feasible_vrp("011001010100")      # van 1 takes two plans
assert not is_feasible_vrp("000001010100")      # van 1 stays idle
assert abs(success_probability({optimal_solutions[0]: 3, "0" * N: 1}, optimal_solutions) - 0.75) < 1e-12
print("helpers OK")


In [ ]:
def analyze_counts_list(
    counts_list, q, best_cost, optimal_solutions, feasibility_fn=is_feasible_vrp
):
    """Summarize a list of Counters (e.g. one per compilation ratio) against the known optimum."""
    summary = []
    for counter in counts_list:
        counter = Counter(counter)
        total = sum(counter.values())
        costs = {bits: qubo_cost(bits, q) for bits in counter}

        avg_gap_cost = (
            sum(count * abs(costs[bits] - best_cost) for bits, count in counter.items()) / total
        )
        optimality_gap = abs(min(costs.values()) - best_cost)
        feasible_shots = sum(count for bits, count in counter.items() if feasibility_fn(bits))

        summary.append(
            {
                "total_shots": total,
                "avg_gap_cost": avg_gap_cost,
                "optimality_gap": optimality_gap,
                "feasible_fraction": feasible_shots / total,
                "n_feasible_distinct": sum(1 for bits in counter if feasibility_fn(bits)),
                "success_probability": success_probability(counter, optimal_solutions),
            }
        )
    return summary


def plot_counts_combined(
    counts_list, q, best_cost, optimal_solutions, x_values=None,
    x_label="Compilation duration ratio",
):
    """Plot, on a single axes, four series as a function of `x_values` (e.g. the compilation
    ratios used to produce each Counter in `counts_list`):

    - average gap cost to optimality, relative to |best_cost|
    - optimality gap of the best sampled string, relative to |best_cost|
    - fraction of feasible shots
    - probability of sampling an optimal solution
    """
    PASQAL_GREEN = "#00A982"
    PASQAL_DARK = "#102A33"
    PASQAL_GRAY = "#8FA6A0"
    PASQAL_BLUE = "#2F6690"

    summary = analyze_counts_list(counts_list, q, best_cost, optimal_solutions)
    x = list(x_values) if x_values is not None else list(range(len(counts_list)))

    # The two gaps are absolute costs -- divided by |best_cost| all four series live on the
    # same 0-1 scale and can share a single axes.
    rel_avg_gap = [row["avg_gap_cost"] / abs(best_cost) for row in summary]
    rel_optimality_gap = [row["optimality_gap"] / abs(best_cost) for row in summary]
    feasible_fraction = [row["feasible_fraction"] for row in summary]
    p_success = [row["success_probability"] for row in summary]

    fig, ax = plt.subplots(figsize=(9, 5.5))
    ax.plot(x, rel_avg_gap, marker="o", color=PASQAL_GREEN, label="Average gap cost / |best cost|")
    ax.plot(x, rel_optimality_gap, marker="s", color=PASQAL_DARK, label="Optimality gap / |best cost|")
    ax.plot(x, feasible_fraction, marker="^", color=PASQAL_GRAY, label="Feasible fraction")
    ax.plot(x, p_success, marker="D", color=PASQAL_BLUE, label="Optimal-solution probability")

    ax.set_xlabel(x_label)
    ax.set_ylabel("Fraction / relative gap")
    ax.set_ylim(0, max([*rel_avg_gap, *rel_optimality_gap, *feasible_fraction, *p_success, 1.0]) * 1.15)
    ax.spines[["top", "right"]].set_visible(False)
    ax.grid(axis="y", alpha=0.18)
    ax.legend(frameon=False, loc="best")
    ax.set_title("Sampling quality vs. compilation ratio", loc="left", weight="bold")
    fig.tight_layout()
    return fig, ax, summary


Plot the four metrics against the compilation ratios and print the summary table.


In [ ]:
# Analysis of the `counts` you collected in section 5 (given).
fig, ax, summary = plot_counts_combined(counts, Q, best_cost, optimal_solutions, x_values=ratios)
plt.show()

for ratio, row in zip(ratios, summary):
    print(
        f"ratio={ratio:.1f}  "
        f"avg_gap_cost={row['avg_gap_cost']:.3f}  "
        f"optimality_gap={row['optimality_gap']:.3f}  "
        f"feasible_fraction={row['feasible_fraction']:.1%}  "
        f"success_probability={row['success_probability']:.1%}"
    )


**Question 6.5** &mdash; interpret the plot:

1. Is there a ratio that works best, or does quality grow monotonically with duration?
2. `feasible_fraction` is always well above `success_probability`. What does that tell you about
   which part of the QUBO the annealer learns first &mdash; the hard constraints or the soft
   congestion costs?
3. `optimality_gap` can be 0 while `success_probability` is only a few percent. Is that good
   enough in practice? What do you do with 1000 shots and a cheap classical cost function?

---

## 7. Retrieve Remote Results *(bonus, needs credentials)*

Only if you submitted cloud jobs in **Bonus 5.3**. Cloud jobs are asynchronous: first check the
status, then retrieve the results and reuse exactly the same analysis as above.

The retrieval cell reads the identifiers back from `my_cloud_jobs.json` rather than from the
`cloud_jobs` list, so it also works in a fresh kernel &mdash; just re-run the `PasqalCloudConnection`
lines of Bonus 5.3 (not the submission loop, or you will queue the sweep a second time).


In [ ]:
# Task: check the status of one of the cloud jobs you submitted.
# status = cloud_jobs[0].get_status()
# print(f"Job status: {status}")


In [ ]:
# Task: retrieve the jobs you submitted in Bonus 5.3 and analyse them with the same plotting
# helper. `my_cloud_jobs.json` is the file written by that cell: a list of
# {"ratio": ..., "job_id": ..., "batch_id": ...} records, in submission order.
#
# import json
# from pathlib import Path
#
# from qoolqit.execution import retrieve_remote_job
#
# JOBS_CACHE_PATH = Path("my_cloud_jobs.json")
#
# with JOBS_CACHE_PATH.open() as f:
#     cached_job_records = json.load(f)
#
# counts_remote = []
# for record in cached_job_records:
#     retrieved_job = retrieve_remote_job(
#         connection, job_id=record["job_id"], batch_id=record["batch_id"]
#     )
#     results = retrieved_job.results()          # blocks until this job is done
#     counts_remote.append(Counter(results.final_bitstrings))
#
# # Each record carries its ratio, so the cloud plot shares the x-axis of the local one.
# fig, ax, summary_remote = plot_counts_combined(
#     counts_remote,
#     Q,
#     best_cost,
#     optimal_solutions,
#     x_values=[record["ratio"] for record in cached_job_records],
# )
# plt.show()
#
# for record, row in zip(cached_job_records, summary_remote):
#     print(
#         f"ratio={record['ratio']:.1f}  "
#         f"avg_gap_cost={row['avg_gap_cost']:.3f}  "
#         f"optimality_gap={row['optimality_gap']:.3f}  "
#         f"feasible_fraction={row['feasible_fraction']:.1%}  "
#         f"success_probability={row['success_probability']:.1%}"
#     )
